In [1]:
import os

# Set the environment variables
os.environ["PYTORCH_NO_CUDA_MEMORY_CACHING"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:25,garbage_collection_threshold:0.5,expandable_segments:True"

# Verify the settings with improved output
print("Environment Variable Settings:")
print(f"  PYTORCH_NO_CUDA_MEMORY_CACHING: {os.environ.get('PYTORCH_NO_CUDA_MEMORY_CACHING', 'Not Set')}")
print(f"  PYTORCH_CUDA_ALLOC_CONF: {os.environ.get('PYTORCH_CUDA_ALLOC_CONF', 'Not Set')}")

print("-" * 30) # add a seperator.

Environment Variable Settings:
  PYTORCH_NO_CUDA_MEMORY_CACHING: 1
  PYTORCH_CUDA_ALLOC_CONF: max_split_size_mb:25,garbage_collection_threshold:0.5,expandable_segments:True
------------------------------


In [2]:
import pickle
import yaml
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import json
import numpy as np
import geneformer
import time
from geneformer import TranscriptomeTokenizer
from itertools import combinations, chain
import scanpy as sc
import anndata as an
from datasets import Dataset, load_from_disk
from scipy.spatial.distance import cdist
import torch
import gc
from importlib import reload
import umap

from geneformer import InSilicoPerturber
from geneformer import InSilicoPerturberStats
from geneformer import EmbExtractor

# local functions
sys.path.append("../../../utils/")
import geneformer_utils as ut

# Load genes

In [3]:
# Load gene annotations
print("[INFO] Loading gene names...")
fpath = "../../../resources/gene_names.tsv.gz"
gdf = pd.read_csv(fpath, sep='\t')
print(f"[DONE] Loaded {len(gdf):,} gene entries.")

# Load list of transcription factors
print("[INFO] Loading transcription factor list...")
fpath = "../../../resources/allTFs_hg38.txt"
tf_list = [x.strip() for x in open(fpath)]
print(f"[DONE] Loaded {len(tf_list):,} transcription factors.")

# Annotate genes
gdf['is_TF'] = gdf['Gene name'].isin(tf_list)
print("[INFO] TF annotation summary:")
print(gdf['is_TF'].value_counts().rename(index={True: 'TF', False: 'Non-TF'}).to_string())

# Preview
gdf.head()


[INFO] Loading gene names...
[DONE] Loaded 73,466 gene entries.
[INFO] Loading transcription factor list...
[DONE] Loaded 1,892 transcription factors.
[INFO] TF annotation summary:
is_TF
Non-TF    71540
TF         1926


/tmp/ipykernel_707012/3722301214.py:4: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  gdf = pd.read_csv(fpath, sep='\t')


,Gene stable ID,Gene description,Gene start (bp),Gene end (bp),Chromosome/scaffold name,Gene type,Gene % GC content,Gene name,is_TF
0,ENSG00000210049,mitochondrially encoded tRNA-Phe (UUU/C) [Sour...,577,647.0,MT,Mt_tRNA,40.85,MT-TF,False
1,ENSG00000211459,mitochondrially encoded 12S rRNA [Source:HGNC ...,648,1601.0,MT,Mt_rRNA,45.49,MT-RNR1,False
2,ENSG00000210077,mitochondrially encoded tRNA-Val (GUN) [Source...,1602,1670.0,MT,Mt_tRNA,42.03,MT-TV,False
3,ENSG00000210082,mitochondrially encoded 16S rRNA [Source:HGNC ...,1671,3229.0,MT,Mt_rRNA,42.81,MT-RNR2,False
4,ENSG00000209082,mitochondrially encoded tRNA-Leu (UUA/G) 1 [So...,3230,3304.0,MT,Mt_tRNA,38.67,MT-TL1,False


In [4]:
all_tfs = gdf[gdf['is_TF']]['Gene stable ID'].to_list()
all_tfs[:10]

['ENSG00000288283',
 'ENSG00000288204',
 'ENSG00000292327',
 'ENSG00000292345',
 'ENSG00000292354',
 'ENSG00000292358',
 'ENSG00000169953',
 'ENSG00000012817',
 'ENSG00000172468',
 'ENSG00000176679']

# Load data

In [5]:
%%time
fpath = "/nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/integrated_anndata/geneformer/pseudotime.dataset"

data = load_from_disk(fpath)
print(f"{data.shape=}")

df = data.to_pandas()
df.head()

data.shape=(15867, 11)
CPU times: user 29.7 ms, sys: 91.5 ms, total: 121 ms
Wall time: 136 ms


,input_ids,batch,phase,n_counts,mean_pseudotime,mean_order,nnz,S_score,G2M_score,cluster_str,length
0,"[2, 10172, 10171, 2839, 2706, 15305, 14422, 27...",hsc,G1,3679,0.808782,0.0,3679,-0.005952,-1.192717,C3,3436
1,"[2, 17488, 17862, 15445, 13841, 17068, 6943, 6...",hsc,G1,3987,0.922000,7688.0,3987,-0.198413,-1.039894,C3,3679
2,"[2, 10172, 13391, 18079, 14211, 283, 1136, 197...",hsc,S,2585,0.264203,9543.0,2585,0.714286,-0.243658,C4,2418
3,"[2, 12826, 18282, 19836, 8699, 2968, 18815, 72...",hsc,G1,2358,0.295864,15150.0,2358,-0.121032,-0.625818,C5,2228
4,"[2, 145, 20006, 19409, 18880, 18408, 11013, 78...",hsc,G1,4995,0.787087,6684.0,4995,-0.444444,-2.328151,C2,4096


# Model Config

In [6]:
model_str = '95m'
params_file = "../../../resources/geneformer_params.yaml"

with open(params_file, 'r') as file:
    params = yaml.safe_load(file)

model = params['models'][model_str]
print(json.dumps(model, indent=2))

{
  "model_path": "/nfs/turbo/umms-indikar/shared/projects/foundation_models/geneformer/Geneformer/gf-12L-95M-i4096/",
  "gene_median": "/nfs/turbo/umms-indikar/shared/projects/foundation_models/geneformer/Geneformer/geneformer/gene_median_dictionary_gc95M.pkl",
  "token_dictionary_file": "/nfs/turbo/umms-indikar/shared/projects/foundation_models/geneformer/Geneformer/geneformer/token_dictionary_gc95M.pkl",
  "gene_mapping_file": "/nfs/turbo/umms-indikar/shared/projects/foundation_models/geneformer/Geneformer/geneformer/ensembl_mapping_dict_gc95M.pkl",
  "model_input_size": 4096,
  "special_token": true
}


In [7]:
print("[INFO] Loading token dictionary...")
with open(model['token_dictionary_file'], 'rb') as f:  
    tokens = pickle.load(f)
print(f"[DONE] Loaded {len(tokens):,} token entries.")

tokens_r = {v: k for k, v in tokens.items()}
print(f"[INFO] Created reverse token map with {len(tokens_r):,} entries.")

print("[INFO] Loading gene mapping...")
with open(model['gene_mapping_file'], 'rb') as f:  
    gene_map = pickle.load(f)
print(f"[DONE] Loaded {len(gene_map):,} gene mappings.")

[INFO] Loading token dictionary...
[DONE] Loaded 20,275 token entries.
[INFO] Created reverse token map with 20,275 entries.
[INFO] Loading gene mapping...
[DONE] Loaded 173,697 gene mappings.


In [8]:
tf_recipe = ['GATA2', 'GFI1B', 'FOS', 'STAT5A', 'REL']

tf_df = []
for tf in tf_recipe:
    ens_id = gene_map[tf]
    token_id = tokens[ens_id]
    tf_df.append((tf, ens_id, token_id))

tf_df = pd.DataFrame(tf_df, columns=["TF", "ensembl_ID", "token_ID"])
print(tf_df.to_string(index=False))

    TF      ensembl_ID  token_ID
 GATA2 ENSG00000179348     14205
 GFI1B ENSG00000165702     11475
   FOS ENSG00000170345     12558
STAT5A ENSG00000126561      5761
   REL ENSG00000162924     10694


# Output directory

In [9]:
outdir = "/scratch/indikar_root/indikar1/shared_data/tmp/"
clear = True

if clear and os.path.exists(outdir):
    print(f"Clearing: {outdir}")
    for f in os.listdir(outdir):
        p = os.path.join(outdir, f)
        if os.path.isdir(p):
            shutil.rmtree(p)
        else:
            os.remove(p)
    print("Done.\n")

Clearing: /scratch/indikar_root/indikar1/shared_data/tmp/
Done.



# Initial embedding

In [10]:
%%time
torch.cuda.empty_cache()

cell_states_to_model = {
    "state_key": "cluster_str", 
    "start_state": "C4", 
    "goal_state": "C3", 
    "alt_states": ["C2", "C1", "C5"],
}

embex = EmbExtractor(
    max_ncells=1000,
    forward_batch_size=10,
    nproc=16,
    token_dictionary_file=model['token_dictionary_file'],
    summary_stat="exact_mean",
)

data_path = "/nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/integrated_anndata/geneformer/pseudotime.dataset"
prefix = 'test'

state_embs_dict = embex.get_state_embs(
    cell_states_to_model,
    model['model_path'], # example 30M fine-tuned model
    data_path,
    outdir,
    prefix,
)

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/84 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

CPU times: user 5min 53s, sys: 17.6 s, total: 6min 11s
Wall time: 6min 10s


In [11]:
# break

In [12]:
# ?InSilicoPerturber

# Set up Perturberations

In [13]:
# Get all row combinations (excluding empty set)
all_combinations = [
    tf_df.iloc[list(idx)].reset_index(drop=True)
    for r in range(1, len(tf_df) + 1)
    for idx in combinations(tf_df.index, r)
]

print(f"Total combinations: {len(all_combinations)}")

Total combinations: 31


# Get Stats

In [14]:
%%time

torch.cuda.empty_cache()

outdir = "/scratch/indikar_root/indikar1/shared_data/tmp/"
isp_prefix = 'isp'

total = len(all_combinations)

for i, perturb in enumerate(all_combinations[10:13]):
    start = time.time()
    genes = ", ".join(perturb['TF'].values)
    gene_str = "-".join(perturb['TF'].values)
    print(f"\n[{i}/{total}] Perturbing: {genes}")

    perturbation_genes = perturb['ensembl_ID'].to_list()
    print(perturbation_genes)

    torch.cuda.empty_cache()
    isp = InSilicoPerturber(
        perturb_type="overexpress",
        perturb_rank_shift=None,
        genes_to_perturb=perturbation_genes,
        anchor_gene=None,
        emb_mode="cls",
        cell_emb_style="mean_pool",
        cell_states_to_model=cell_states_to_model,
        state_embs_dict=state_embs_dict,
        max_ncells=100,
        emb_layer=0,
        forward_batch_size=10,
        nproc=16,
        token_dictionary_file=model['token_dictionary_file'],
    )

    print(f"\tInSilicoPerturber set up complete")
    isp.perturb_data(
        model['model_path'], 
        data_path,
        outdir,
        isp_prefix,
    )

    print(f"\tisp.perturb_data complete")

    ispstats = InSilicoPerturberStats(
        mode="goal_state_shift",
        genes_perturbed=perturbation_genes,
        cell_states_to_model=cell_states_to_model,
        token_dictionary_file=model['token_dictionary_file'],
        gene_name_id_dictionary_file=model['gene_mapping_file'],
    )

    print(f"\tGetting stats...")

    # extracts data from intermediate files and processes stats to output in final .csv
    ispstats.get_stats(
        outdir,
        None,
        outdir,
        f"{gene_str}_stats",
    )

    elapsed = time.time() - start
    print(f"\tDone in {elapsed:.2f} seconds.")


[0/31] Perturbing: GFI1B, STAT5A
['ENSG00000165702', 'ENSG00000126561']
	InSilicoPerturber set up complete


  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

...

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

	isp.perturb_data complete
	Getting stats...


  0%|          | 0/3 [00:00<?, ?it/s]

	Done in 105.16 seconds.
CPU times: user 3min 22s, sys: 8.6 s, total: 3min 30s
Wall time: 5min 9s


In [15]:
?ispstats.get_stats

Signature:
ispstats.get_stats(
    input_data_directory,
    null_dist_data_directory,
    output_directory,
    output_prefix,
    null_dict_list=None,
)
Docstring:
Get stats for in silico perturbation data and save as results in output_directory.

**Parameters:**

input_data_directory : Path
    | Path to directory containing cos_sim dictionary inputs
null_dist_data_directory : Path
    | Path to directory containing null distribution cos_sim dictionary inputs
output_directory : Path
    | Path to directory where perturbation data will be saved as .csv
output_prefix : str
    | Prefix for output .csv
null_dict_list: list[dict]
    | List of loaded null distribution dictionary if more than one comparison vs. the null is to be performed

**Outputs:**

Definition of possible columns in .csv output file.

| Of note, not all columns will be present in all output files.
| Some columns are specific to particular perturbation modes.

| "Gene": gene token
| "Gene_name": gene name
| "Ensembl_I

In [16]:
print('done')

done


In [17]:
# fpath = f"{outdir}stats.csv"
# # df = pd.read_csv(fpath)
# df.head()

In [18]:
break

SyntaxError: 'break' outside loop (668683560.py, line 1)

### in silico perturbation in deletion mode to determine genes whose deletion in the dilated cardiomyopathy (dcm) state significantly shifts the embedding towards non-failing (nf) state

In [ ]:
# first obtain start, goal, and alt embedding positions
# this function was changed to be separate from perturb_data
# to avoid repeating calcuations when parallelizing perturb_data
cell_states_to_model={"state_key": "disease", 
                      "start_state": "dcm", 
                      "goal_state": "nf", 
                      "alt_states": ["hcm"]}

filter_data_dict={"cell_type":["Cardiomyocyte1","Cardiomyocyte2","Cardiomyocyte3"]}

# OF NOTE: token_dictionary_file must be set to the gc-30M token dictionary if using a 30M series model
# (otherwise the EmbExtractor will use the current default model dictionary)
# 30M token dictionary: https://huggingface.co/ctheodoris/Geneformer/blob/main/geneformer/gene_dictionaries_30m/token_dictionary_gc30M.pkl
embex = EmbExtractor(model_type="CellClassifier", # if using previously fine-tuned cell classifier model
                     num_classes=3,
                     filter_data=filter_data_dict,
                     max_ncells=1000,
                     emb_layer=0,
                     summary_stat="exact_mean",
                     forward_batch_size=256,
                     nproc=16)

state_embs_dict = embex.get_state_embs(cell_states_to_model,
                                       "../fine_tuned_models/gf-6L-30M-i2048_CellClassifier_cardiomyopathies_220224", # example 30M fine-tuned model
                                       "path/to/input_data",
                                       "path/to/output_directory",
                                       "output_prefix")

In [ ]:
# OF NOTE: token_dictionary_file must be set to the gc-30M token dictionary if using a 30M series model
# (otherwise the InSilicoPerturber will use the current default model dictionary)
# 30M token dictionary: https://huggingface.co/ctheodoris/Geneformer/blob/main/geneformer/gene_dictionaries_30m/token_dictionary_gc30M.pkl
isp = InSilicoPerturber(perturb_type="delete",
                        perturb_rank_shift=None,
                        genes_to_perturb="all",
                        combos=0,
                        anchor_gene=None,
                        model_type="CellClassifier", # if using previously fine-tuned cell classifier model
                        num_classes=3,
                        emb_mode="cell",
                        cell_emb_style="mean_pool",
                        filter_data=filter_data_dict,
                        cell_states_to_model=cell_states_to_model,
                        state_embs_dict=state_embs_dict,
                        max_ncells=2000,
                        emb_layer=0,
                        forward_batch_size=400,
                        nproc=16)

In [ ]:
# outputs intermediate files from in silico perturbation

isp.perturb_data("../fine_tuned_models/gf-6L-30M-i2048_CellClassifier_cardiomyopathies_220224", # example 30M fine-tuned model
                 "path/to/input_data",
                 "path/to/isp_output_directory",
                 "output_prefix")

In [ ]:
# OF NOTE: token_dictionary_file must be set to the gc-30M token dictionary if using a 30M series model
# (otherwise the InSilicoPerturberStats will use the current default model dictionary)
# 30M token dictionary: https://huggingface.co/ctheodoris/Geneformer/blob/main/geneformer/gene_dictionaries_30m/token_dictionary_gc30M.pkl
ispstats = InSilicoPerturberStats(mode="goal_state_shift",
                                  genes_perturbed="all",
                                  combos=0,
                                  anchor_gene=None,
                                  cell_states_to_model=cell_states_to_model)

In [ ]:
# extracts data from intermediate files and processes stats to output in final .csv
ispstats.get_stats("path/to/isp_output_directory", # this should be the directory 
                   None,
                   "path/to/isp_stats_output_directory",
                   "output_prefix")